In [0]:
%pip install mlforecast lightgbm numba

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np

df = spark.table("workspace.default.features").toPandas()
df["visit_date"] = pd.to_datetime(df["visit_date"])

data = df[["air_store_id", "visit_date", "visitors"]].rename(
    columns={"air_store_id": "unique_id", "visit_date": "ds", "visitors": "y"}
)
print(data.shape)
data.head()

(228904, 3)


,unique_id,ds,y
0,air_dfe068a1bf85f395,2017-03-08,51
1,air_dfe068a1bf85f395,2017-03-09,40
2,air_dfe068a1bf85f395,2017-03-10,47
3,air_dfe068a1bf85f395,2017-03-11,32
4,air_dfe068a1bf85f395,2017-03-12,24


In [0]:
from utilsforecast.preprocessing import fill_gaps

data = fill_gaps(data, freq="D")

data["y"] = data["y"].fillna(0)

print("rows after gap-filling:", data.shape)
print("missing y:", data["y"].isna().sum())
data.head()

rows after gap-filling: (267632, 3)
missing y: 0


,unique_id,ds,y
0,air_00a91d42b08b08d9,2016-08-04,30.0
1,air_00a91d42b08b08d9,2016-08-05,42.0
2,air_00a91d42b08b08d9,2016-08-06,7.0
3,air_00a91d42b08b08d9,2016-08-07,0.0
4,air_00a91d42b08b08d9,2016-08-08,27.0


In [0]:
import lightgbm as lgb
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences  
from mlforecast.lag_transforms import RollingMean, RollingStd
from numba import njit  


fcst = MLForecast(
    models={"lgbm": lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=31, random_state=42
    )},
    freq="D",                          
    lags=[1, 7, 14, 28],               
    lag_transforms={                   
        7:  [RollingMean(7),  RollingStd(7)],
        28: [RollingMean(28), RollingStd(28)],
    },
    date_features=["dayofweek", "month"],  
)

In [0]:
cv_results = fcst.cross_validation(
    df=data,
    n_windows=3,
    h=35,
)
cv_results.head()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/utilsforecast/processing.py:806: UserWarning: The following series are too short for the window and will be dropped: ['air_1c0b150f9e696a5f', 'air_28dbe91c4c9656be', 'air_2a485b92210c98b5', 'air_31c753b48a657b6c', 'air_6b65745d432fd77f', 'air_7420042ff75f9aca', ...]
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:683: UserWarning: The following series were dropped completely due to the transformations and features: ['air_4b55d8aea1d2b395', 'air_52a08ef3efdb4bb0', 'air_8d61f49aa0373492', 'air_a257c9749d8d0ff6', 'air_df554c4527a1cfe6', 'air_e00fe7853c0100d6', ...].
These series won't show up if you use `MLForecast.forecast_fitted_values()`.
You can set `dropna=False` or use transformations that require less samples to mitigate this
  warnings.warn(


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006958 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1605
[LightGBM] [Info] Number of data points in the train set: 136906, number of used features: 10
[LightGBM] [Info] Start training from score 17.868669


/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag28, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag28, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag28, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007232 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1620
[LightGBM] [Info] Number of data points in the train set: 165249, number of used features: 10
[LightGBM] [Info] Start training from score 17.662503


/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, rolling_std_lag7_window_size7, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, rolling_std_lag7_window_size7, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, ro

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.007718 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1631
[LightGBM] [Info] Number of data points in the train set: 193710, number of used features: 10
[LightGBM] [Info] Start training from score 17.716158


/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, rolling_std_lag7_window_size7, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, rolling_std_lag7_window_size7, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, ro

,unique_id,ds,cutoff,y,lgbm
0,air_00a91d42b08b08d9,2017-01-08,2017-01-07,0.0,0.547636
1,air_00a91d42b08b08d9,2017-01-09,2017-01-07,0.0,6.623093
2,air_00a91d42b08b08d9,2017-01-10,2017-01-07,37.0,13.092658
3,air_00a91d42b08b08d9,2017-01-11,2017-01-07,27.0,16.411232
4,air_00a91d42b08b08d9,2017-01-12,2017-01-07,15.0,19.032096


In [0]:
from sklearn.metrics import mean_squared_error

def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

scored = cv_results[cv_results["y"] > 0].copy()

per_fold = (
    scored
    .groupby("cutoff")
    .apply(lambda g: rmsle(g["y"].values, g["lgbm"].values), include_groups=False)
    .rename("rmsle")
)

print("RMSLE per fold (open days only):")
print(per_fold)
print(f"\nMean RMSLE across folds: {per_fold.mean():.4f}")
print(f"Std  RMSLE across folds: {per_fold.std():.4f}")


RMSLE per fold (open days only):
cutoff
2017-01-07    0.670608
2017-02-11    0.535511
2017-03-18    0.575574
Name: rmsle, dtype: float64

Mean RMSLE across folds: 0.5939
Std  RMSLE across folds: 0.0694


## Time-series cross-validation (3 rolling windows, 35-day horizon)

Scored on genuinely open days only (closed days filled as 0 for feature-building,
excluded from the metric, since RMSLE over-penalises predictions on closed days).

| Fold (cutoff) | RMSLE |
|---------------|-------|
| 2017-01-07 | 0.671 |
| 2017-02-11 | 0.536 |
| 2017-03-18 | 0.576 |
| **Mean ± Std** | **0.594 ± 0.069** |

**Read:** robust RMSLE of **0.594 ± 0.069** across three periods — comfortably beats the
0.6683 baseline on an averaged (harder, more honest) basis than a single split. The
January fold is hardest, consistent with New-Year holiday disruption; the model is
otherwise stable across time. Holiday periods are the clear weak spot.

In [0]:
import pandas as pd
import numpy as np

df = spark.table("workspace.default.features").toPandas()
df["visit_date"] = pd.to_datetime(df["visit_date"])

data = df[["air_store_id","visit_date","visitors"]].rename(
    columns={"air_store_id":"unique_id","visit_date":"ds","visitors":"y"})

from utilsforecast.preprocessing import fill_gaps      
data = fill_gaps(data, freq="D", id_col="unique_id", time_col="ds")
data["y"] = data["y"].fillna(0)

val_start = data["ds"].max() - pd.Timedelta(days=35)
train = data[data["ds"] <  val_start]
val   = data[data["ds"] >= val_start]
print("train:", train.shape, "| val:", val.shape)

train: (237887, 3) | val: (29745, 3)


In [0]:
quantile_models = {
    "p10": lgb.LGBMRegressor(objective="quantile", alpha=0.1,
                             n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1),
    "p50": lgb.LGBMRegressor(objective="quantile", alpha=0.5,
                             n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1),
    "p90": lgb.LGBMRegressor(objective="quantile", alpha=0.9,
                             n_estimators=500, learning_rate=0.05, random_state=42, verbosity=-1),
}

fcst = MLForecast(
    models=quantile_models,
    freq="D",
    lags=[1, 7, 14, 28],
    lag_transforms={7:[RollingMean(7),RollingStd(7)], 28:[RollingMean(28),RollingStd(28)]},
    date_features=["dayofweek","month"],
)

fcst.fit(train)          
preds = fcst.predict(h=35)   
preds.head()

/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:683: UserWarning: The following series were dropped completely due to the transformations and features: ['air_28dbe91c4c9656be', 'air_2a485b92210c98b5', 'air_6b65745d432fd77f', 'air_7420042ff75f9aca', 'air_8110d68cc869b85e', 'air_8e8f42f047537154', ...].
These series won't show up if you use `MLForecast.forecast_fitted_values()`.
You can set `dropna=False` or use transformations that require less samples to mitigate this
  warnings.warn(
/local_disk0/.ephemeral_nfs/envs/pythonEnv-3fdc5de9-0e36-4e1d-8421-8b99f183b58a/lib/python3.12/site-packages/mlforecast/core.py:1030: UserWarning: Found null values in lag7, lag14, lag28, rolling_mean_lag7_window_size7, rolling_std_lag7_window_size7, rolling_mean_lag28_window_size28, rolling_std_lag28_window_size28.
  warnings.warn(f'Found null values in {", ".join(cols_with_nulls)}.')
/local_disk0/.ephemeral_nfs/envs/pythonEn

,unique_id,ds,p10,p50,p90
0,air_00a91d42b08b08d9,2017-03-18,7.925949,2.006861e+01,38.741749
1,air_00a91d42b08b08d9,2017-03-19,0.000000,1.021854e-10,1.576406
2,air_00a91d42b08b08d9,2017-03-20,0.000000,2.100007e+01,36.516870
3,air_00a91d42b08b08d9,2017-03-21,0.000000,2.886029e+01,41.219689
4,air_00a91d42b08b08d9,2017-03-22,0.000000,3.256400e+01,44.647096


In [0]:
def rmsle(y_true, y_pred):
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred)))

ev = val.merge(preds, on=["unique_id","ds"], how="inner")
ev = ev[ev["y"] > 0].copy()

print("P50 RMSLE:", round(rmsle(ev["y"].values, ev["p50"].values), 4),
      " (untuned CV mean was ~0.594)")

P50 RMSLE: 0.622  (untuned CV mean was ~0.594)


In [0]:
inside = (ev["y"] >= ev["p10"]) & (ev["y"] <= ev["p90"])
coverage = inside.mean()

below_p10 = (ev["y"] < ev["p10"]).mean()
above_p90 = (ev["y"] > ev["p90"]).mean()

print("Calibration of the P10-P90 band (target ~80% coverage):")
print(f"  coverage (inside band): {coverage:.1%}")
print(f"  below P10:              {below_p10:.1%}   (target ~10%)")
print(f"  above P90:              {above_p90:.1%}   (target ~10%)")

Calibration of the P10-P90 band (target ~80% coverage):
  coverage (inside band): 93.4%
  below P10:              0.9%   (target ~10%)
  above P90:              5.8%   (target ~10%)


## P10–P90 calibration
- P10–P90 coverage: **93.4%** (target ~80%) → bands are **underconfident / too wide**.
- Below P10: 0.9% (target 10%) — the P10 is far too low and rarely binds.
- Above P90: 5.8% (target 10%) — the P90 is mildly too high.
- **Read:** the model errs toward caution (wide bands) rather than false precision — the
  safer failure mode. Tightening would use narrower target quantiles (e.g. P20/P80) or
  conformal calibration on a held-out set.